In [1]:
import pandas as pd

train_df = pd.read_csv("../data/processed/train_processed.csv")

print(train_df.columns.tolist())

['rating', 'restaurantRating', 'rating_deviation', 'friendCount', 'reviewCount', 'reviewUsefulCount', 'usefulCount', 'coolCount', 'funnyCount', 'complimentCount', 'tipCount', 'fanCount', 'reviewer_account_age_days', 'reviewer_engagement_ratio', 'reviewer_social_reach', 'reviewer_helpfulness_ratio', 'reviewer_activity_level', 'review_word_count', 'flagged']


In [1]:
# =====================================================
# Fake Review Detector - Streamlit Application
# =====================================================

import streamlit as st
import pandas as pd
import numpy as np
import joblib
from pathlib import Path


# =====================================================
# Project Paths
# =====================================================

BASE_DIR = Path(__file__).resolve().parent.parent

MODEL_PATH = BASE_DIR / "models" / "final_random_forest_model.pkl"
FEATURE_LIST_PATH = BASE_DIR / "models" / "final_feature_list.pkl"
THRESHOLD_PATH = BASE_DIR / "models" / "final_threshold.pkl"


# =====================================================
# Page Configuration
# =====================================================

st.set_page_config(
    page_title="Fake Review Detector",
    layout="wide",
    initial_sidebar_state="expanded"
)


# =====================================================
# Load Model Files
# =====================================================

@st.cache_resource
def load_model_files():
    model = joblib.load(MODEL_PATH)
    feature_list = joblib.load(FEATURE_LIST_PATH)
    threshold = joblib.load(THRESHOLD_PATH)

    return model, feature_list, threshold


model, feature_list, threshold = load_model_files()


# =====================================================
# Application Header
# =====================================================

st.title("Fake Review Detector")
st.write(
    "This application detects whether a review is genuine or fake "
    "using a Random Forest machine learning model."
)

st.info(f"Final Model Used: Random Forest Classifier | Threshold Used: {threshold}")


# =====================================================
# Sidebar Information
# =====================================================

st.sidebar.header("Project Information")

st.sidebar.write("Project Name: Fake Review Detector")
st.sidebar.write("Model: Random Forest Classifier")
st.sidebar.write("Threshold: 0.40")
st.sidebar.write("Dataset: Yelp Fake Review Dataset")

st.sidebar.header("Prediction Labels")
st.sidebar.write("0 = Genuine Review")
st.sidebar.write("1 = Fake Review")


# =====================================================
# User Input Section
# =====================================================

st.header("Enter Review and Reviewer Details")

col1, col2 = st.columns(2)

with col1:
    rating = st.slider(
        "Review Rating",
        min_value=1,
        max_value=5,
        value=5
    )

    review_text = st.text_area(
        "Review Text",
        "This place was amazing and the service was very good."
    )

    review_count = st.number_input(
        "Reviewer Total Review Count",
        min_value=0,
        value=10
    )

    useful_count = st.number_input(
        "Useful Count",
        min_value=0,
        value=5
    )

    cool_count = st.number_input(
        "Cool Count",
        min_value=0,
        value=2
    )

with col2:
    funny_count = st.number_input(
        "Funny Count",
        min_value=0,
        value=1
    )

    friend_count = st.number_input(
        "Friend Count",
        min_value=0,
        value=20
    )

    fan_count = st.number_input(
        "Fan Count",
        min_value=0,
        value=3
    )

    tip_count = st.number_input(
        "Tip Count",
        min_value=0,
        value=4
    )

    reviewer_account_age_days = st.number_input(
        "Reviewer Account Age in Days",
        min_value=0,
        value=500
    )


# =====================================================
# Feature Engineering
# =====================================================

review_word_count = len(review_text.split())

reviewer_engagement_ratio = (
    useful_count + cool_count + funny_count
) / (review_count + 1)

reviewer_social_reach = friend_count + fan_count

reviewer_helpfulness_ratio = useful_count / (review_count + 1)

reviewer_activity_level = review_count + tip_count

rating_deviation = abs(rating - 3.5)


# =====================================================
# Prepare Input Data
# =====================================================

input_data = pd.DataFrame({
    "rating": [rating],
    "review_word_count": [review_word_count],
    "reviewer_engagement_ratio": [reviewer_engagement_ratio],
    "reviewer_social_reach": [reviewer_social_reach],
    "reviewer_helpfulness_ratio": [reviewer_helpfulness_ratio],
    "reviewer_activity_level": [reviewer_activity_level],
    "rating_deviation": [rating_deviation],
    "reviewer_account_age_days": [reviewer_account_age_days]
})


# =====================================================
# Align Input Features with Final Feature List
# =====================================================

for feature in feature_list:
    if feature not in input_data.columns:
        input_data[feature] = 0

input_data = input_data[feature_list]


# =====================================================
# Display Calculated Features
# =====================================================

st.subheader("Calculated Behavioral Features")
st.dataframe(input_data, width="stretch")


# =====================================================
# Prediction Section
# =====================================================

if st.button("Predict Review"):

    prediction_probability = model.predict_proba(input_data)[0][1]

    if prediction_probability >= threshold:
        prediction = 1
    else:
        prediction = 0

    st.subheader("Prediction Result")

    if prediction == 1:
        st.error("The review is predicted as Fake Review")
    else:
        st.success("The review is predicted as Genuine Review")

    st.write("Fake Review Probability:", round(prediction_probability, 4))
    st.write("Threshold Used:", threshold)

    genuine_probability = 1 - prediction_probability

    st.write("Genuine Review Probability:", round(genuine_probability, 4))


# =====================================================
# Footer
# =====================================================

st.markdown("---")
st.write("Developed as a Machine Learning project for fake review detection.")

NameError: name '__file__' is not defined